# Урок 30. AI: SAM — сегментация окон на фото с дрона

**Что делает этот ноутбук**

1. Берёт фото домов с камеры дрона.
2. По текстовому запросу `"window."` модель **Grounding DINO** находит прямоугольники (bounding boxes) всех окон.
3. Модель **SAM** превращает каждый прямоугольник в точную маску окна.
4. Все маски склеиваются в одну чёрно-белую картинку `mask.jpg` — это и есть ответ.

**Перед запуском:** `Среда выполнения` → `Сменить среду выполнения` → `T4 GPU`.
На CPU тоже работает, просто медленнее (5–10 минут вместо 40 секунд).

Запускать ячейки строго сверху вниз: `Shift + Enter`.

## Шаг 0. Установка библиотек

`transformers` — библиотека Hugging Face, в ней уже лежат и Grounding DINO, и SAM.
Ставим свежую версию, чтобы Grounding DINO точно был внутри.

In [ ]:
!pip install -q -U transformers
print("Готово")

## Шаг 1. Загружаем фото с дрона

**Вариант А (рекомендуется).** Скачайте картинку из задания на компьютер,
затем запустите ячейку ниже и выберите файл — он загрузится в Colab.

**Вариант Б.** Если у вас есть прямая ссылка на картинку из задания,
раскомментируйте строку с `!wget` во второй ячейке и вставьте ссылку.

⚠️ Важно: используйте **оригинальный** файл из задания, не пересохранённый
скриншот. Маска-ответ должна иметь ровно тот же размер в пикселях,
что и исходное фото.

In [ ]:
from google.colab import files

uploaded = files.upload()          # откроется окно выбора файла
IMAGE_PATH = list(uploaded.keys())[0]
print("Файл загружен:", IMAGE_PATH)

In [ ]:
# Вариант Б: скачать по прямой ссылке (раскомментируйте и подставьте свою ссылку)
# !wget -q -O drone_photo.png "ВСТАВЬТЕ_ССЫЛКУ_ИЗ_ЗАДАНИЯ"
# IMAGE_PATH = "drone_photo.png"
# print("Файл скачан:", IMAGE_PATH)

## Шаг 2. Импорты и настройки

Здесь собраны все «ручки», которые можно крутить, если результат не понравится.

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import SamProcessor, SamModel

TEXT_PROMPT = "window."          # текстовый промпт; точка в конце обязательна
BOX_THRESHOLD = 0.30             # порог уверенности детектора
TEXT_THRESHOLD = 0.25            # порог соответствия бокса словам промпта

GDINO_MODEL_ID = "IDEA-Research/grounding-dino-base"
SAM_MODEL_ID = "facebook/sam-vit-huge"

OUTPUT_MASK_PATH = "mask.jpg"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", DEVICE)

image = Image.open(IMAGE_PATH).convert("RGB")
print("Размер изображения:", image.size)   # (ширина, высота)
image

## Шаг 3. Grounding DINO: текст → прямоугольники

Модель скачается автоматически (~700 МБ), первый запуск занимает 1–2 минуты.

In [ ]:
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID)
gdino_model = gdino_model.to(DEVICE).eval()

inputs = gdino_processor(images=image, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = gdino_model(**inputs)

# Пороги передаём БЕЗ имён (позиционно): имя третьего аргумента менялось
# между версиями transformers, а его позиция — нет.
results = gdino_processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    BOX_THRESHOLD,
    TEXT_THRESHOLD,
    target_sizes=[(image.size[1], image.size[0])],   # (высота, ширина)
)

boxes = results[0]["boxes"].detach().cpu().numpy()
scores = results[0]["scores"].detach().cpu().numpy()

print("Найдено окон:", len(boxes))
print("Уверенность: min = %.3f, max = %.3f" % (scores.min(), scores.max()))

## Шаг 4. Смотрим глазами, что нашёл детектор

Это проверка «на здравый смысл». Если рамок слишком мало — понизьте `BOX_THRESHOLD`.
Если в рамки попали двери, крыши, солнечные панели — повысьте.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(11, 11))
ax.imshow(image)
for x0, y0, x1, y1 in boxes:
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                   linewidth=1.6, edgecolor="lime", facecolor="none"))
ax.set_title(f"Grounding DINO: {len(boxes)} рамок")
ax.axis("off")
plt.show()

## Шаг 5. SAM: прямоугольники → точные маски

Модель скачается автоматически (~2.4 ГБ). SAM один раз кодирует картинку,
а потом очень быстро отвечает на каждый прямоугольник-промпт.

In [ ]:
sam_processor = SamProcessor.from_pretrained(SAM_MODEL_ID)
sam_model = SamModel.from_pretrained(SAM_MODEL_ID).to(DEVICE).eval()

# Внешний список — это батч картинок. У нас в батче одна картинка,
# внутри неё список всех её прямоугольников.
input_boxes = [[[float(v) for v in box] for box in boxes]]

sam_inputs = sam_processor(image, input_boxes=input_boxes, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    sam_outputs = sam_model(**sam_inputs, multimask_output=False)

masks = sam_processor.image_processor.post_process_masks(
    sam_outputs.pred_masks.detach().cpu(),
    sam_inputs["original_sizes"].cpu(),
    sam_inputs["reshaped_input_sizes"].cpu(),
)[0]

masks = masks.squeeze(1).numpy().astype(bool)   # (N, H, W)
print("Форма набора масок:", masks.shape)

## Шаг 6. Склеиваем все маски в одну и сохраняем ответ

`np.any(masks, axis=0)` = «пиксель белый, если он попал хотя бы в одну маску».

In [ ]:
mask = np.any(masks, axis=0)                 # (H, W), значения True / False
print("Доля белых пикселей: %.2f%%" % (mask.mean() * 100))

mask_uint8 = mask.astype(np.uint8) * 255     # True -> 255 (белый), False -> 0 (чёрный)
mask_pil = Image.fromarray(mask_uint8)
mask_pil.save(OUTPUT_MASK_PATH)

print("Сохранено:", OUTPUT_MASK_PATH, "| размер:", mask_pil.size, "| режим:", mask_pil.mode)

## Шаг 7. Самопроверка глазами

Слева — сама маска (её и сдаём), справа — маска, наложенная на фото.
На маске не должно быть никаких подписей, рамок и вотермарок — только чёрное и белое.

In [ ]:
overlay = np.array(image).astype(np.float32)
overlay[mask] = 0.45 * overlay[mask] + 0.55 * np.array([255.0, 40.0, 40.0])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(mask_uint8, cmap="gray")
axes[0].set_title("mask.jpg — это ответ")
axes[0].axis("off")
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title("Проверка: окна подсвечены")
axes[1].axis("off")
plt.show()

## Шаг 8. Скачиваем файл ответа

In [ ]:
files.download(OUTPUT_MASK_PATH)

## Что делать, если баллов мало

Крутите ручки **по одной** и каждый раз пересматривайте картинку из шага 4.

| Симптом | Что менять |
|---|---|
| Окон найдено заметно меньше, чем видно глазами | `BOX_THRESHOLD` = 0.25 → 0.20 → 0.15 |
| В маску попали двери, крыши, солнечные панели, бассейн | `BOX_THRESHOLD` = 0.35 → 0.40 |
| Одно окно «разлилось» на всю стену | это ошибка SAM на большом боксе — поднимите `BOX_THRESHOLD` |
| Совсем ничего не найдено | проверьте, что промпт заканчивается точкой: `"window."` |
| Хочется чуть другой набор объектов | промпт `"window"`, `"windows"`, `"window . glass window ."` |
| Нужны маски поаккуратнее / побыстрее | `SAM_MODEL_ID` = `facebook/sam-vit-base` (быстрее) |

После каждого изменения перезапускайте шаги 3 → 7 и заново скачивайте `mask.jpg`.